# Vector stores and semantic search



In [1]:
from sentence_transformers import SentenceTransformer, util
import pandas as pd
import numpy as np

c:\Users\antag\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Part I: Basic vector store implementation

In [2]:
import numpy as np

class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.model = embedding_model
        self.documents = []
        self.embeddings = []

    def add_documents(self, documents: list[Document]):
        self.documents.extend(documents)
        new_embs = self.model.encode([doc.text for doc in documents], convert_to_numpy=True)
        self.embeddings.extend(new_embs)

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        if not self.documents:
            return []
        query_emb = self.model.encode([query], convert_to_numpy=True)
        scores = util.cos_sim(query_emb, self.embeddings)[0].numpy()
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [SearchResult(float(scores[i]), self.documents[i]) for i in top_idx]

In [3]:
df = pd.read_csv("Animal_Fun_Facts/animal-fun-facts-dataset.csv").fillna("")

documents = [
    Document(
        text=row["text"],
        metadata={
            "animal_name": row["animal_name"],
            "source":       row["source"],
            "media_link":   row["media_link"],
            "wikipedia_link": row["wikipedia_link"],
        }
    )
    for _, row in df.iterrows()
    if row["text"].strip()       
]

print(f"Documentos cargados: {len(documents)}")

Documentos cargados: 7731


## Part II: Filtering by metadata

In [4]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        pass

    def add_documents(self, documents: list[Document]):
        pass

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        pass